# Homework 2: Vector Search

Questions: [homework.md](../../course/cohorts/2026/homework/02-vector-search/homework.md)

## Setup

In [14]:
from embedder import Embedder

embed = Embedder()

## Q1. Embedding a query

In [15]:
question = "How does approximate nearest neighbor search work?"

question_vector = embed.encode(question)
question_vector[0]

np.float64(-0.020582036807885073)

## Loading the data

In [16]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [17]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Q2. Cosine similarity

In [25]:
document = next(
    document for document in documents
    if document['filename'] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

content_vector = embed.encode(document["content"])

content_vector.dot(question_vector)

np.float64(0.361070280302606)

## Q3. Chunking and search by hand

In [19]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [20]:
chunks[0], chunks[1]

({'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [37]:
texts = [chunk["content"] for chunk in chunks]

In [38]:
X = embed.encode_batch(texts)

In [39]:
X[0][0]

np.float64(-0.08756473320658477)

In [40]:
X.shape

(295, 384)

In [41]:
scores = X.dot(question_vector)

In [42]:
import numpy as np

In [43]:
top5 = np.argsort(-scores)[:5]

In [44]:
top5

array([ 94,  14, 162,  85, 253])

In [46]:
for idx in top5:
    print(chunks[idx]['filename'])

02-vector-search/lessons/07-sqlitesearch-vector.md
01-agentic-rag/lessons/05-search.md
04-evaluation/lessons/05-search-metrics.md
02-vector-search/lessons/04-vector-search.md
06-best-practices/lessons/03-reranking.md


## Q4. Vector search with minsearch

In [47]:
query_2 = "What metric do we use to evaluate a search engine?"

In [48]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

In [52]:
query_2_vector = embed.encode(query_2)

query_2_results = vindex.search(query_2_vector, num_results=5)

In [53]:
query_2_results[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

## Q5. Text search vs vector search

In [63]:
query_3 = "How do I store vectors in PostgreSQL?"
query_3_vector = embed.encode(query_3)

In [64]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

In [65]:
vector_files = {result['filename'] for result in vindex.search(query_3_vector, num_results=5)}
text_files = {result['filename'] for result in index.search(query_3, num_results=5)}

vector_files - text_files

{'02-vector-search/lessons/08-pgvector.md'}

## Q6. Hybrid search